# 🏛️ Entrenar HyperNCD (M2) con ArCH · con medición de tiempo, RAM y GPU

**Qué hace:** convierte ArCH (ya descargado en tu Drive) a bloques de 10 m, y entrena HyperNCD con las 15 escenas de `Training`, validando con las 2 de `Test`. **Cada comando pesado corre dentro de `monitor_recursos.py`**, que mide tiempo total, RAM, memoria y uso de GPU y deja un gráfico y un json.

**Clases:** 9 de ArCH (`other` se ignora). Nuevas (las que descubre sin etiquetas): `arch`, `column`, `vault`. Conocidas: `moldings`, `floor`, `door_window`, `wall`, `stairs`, `roof`. Se cambia en `hyperncd/config/arch_dataset.yaml` (`novel_classes`).

**Orden:** 1 a 3 preparan. 4 convierte (una sola vez). 5 es una **prueba de 1 época** (opcional si ya la corriste). 6 es la **corrida completa (30 épocas)**, reanudable. 7 muestra la medición. 8 **predice tu capilla etiquetada** con el mejor checkpoint y con el último. 9 empaqueta logs y predicciones para bajarlos y calcular las métricas en local.

**Requisitos:** GPU, `hyperncd_env_cache.tar.gz` en tu Drive (cuaderno 1), y que los cambios de este experimento estén subidos a GitHub (la celda 2 lo comprueba).

In [ ]:
# @title 1. GPU, Drive y parámetros
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, glob, json
from google.colab import drive
drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/PFC1_HyperNCD_Sonata"
ARCH = f"{DRIVE}/arch"
CRUDO = f"{ARCH}/datos/ArCH_labelled"          # los .txt descargados
TILES = f"{ARCH}/tiles"                        # bloques convertidos (se crean en la celda 4)
SAL = f"{ARCH}/entrenamiento"                  # checkpoints, logs y mediciones de esta corrida
BRANCH = "main"                                # @param {type:"string"}
EPOCAS = 30                                    # @param {type:"integer"}  (celda 6; la celda 5 usa 1)
BATCH = 4                                      # @param {type:"integer"}
WORKERS = 0                                    # @param {type:"integer"}  (0: con 2 el DataLoader se colgaba en Colab)
for d in (TILES, SAL): os.makedirs(d, exist_ok=True)
assert glob.glob(f"{CRUDO}/Training/*.txt"), f"No encuentro los .txt de ArCH en {CRUDO}"
print("ArCH:", len(glob.glob(f"{CRUDO}/Training/*.txt")), "escenas Training +", len(glob.glob(f"{CRUDO}/Test/*.txt")), "Test")

In [ ]:
# @title 2. Clonar el repo y comprobar que están los cambios de ArCH
import os
REPO_OWNER, REPO_NAME = "manyorycuevamendoza", "PFC1-HyperNCD-Sonata"
REPO = f"/content/{REPO_NAME}"
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
if not token:
    from getpass import getpass
    token = getpass("GitHub token (oculto): ").strip()
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO}
else:
    !git -C {REPO} pull -q origin {BRANCH}
necesarios = ["experimentacion/arch_entrenamiento/convertir_arch.py", "experimentacion/monitor_recursos/monitor_recursos.py",
              "hyperncd/config/arch_dataset.yaml", "hyperncd/train.py"]
faltan = [f for f in necesarios if not os.path.exists(f"{REPO}/{f}")]
assert not faltan, f"Faltan en GitHub (hay que subirlos): {faltan}"
assert "ArCH" in open(f"{REPO}/hyperncd/utils/dataset.py").read(), "dataset.py en GitHub aún no tiene ArCH: súbelo"
print("✅ repo listo")

In [ ]:
# @title 2b. Parches de compatibilidad (solo se aplican si el repo de GitHub aún no los trae)
# El entorno de Colab usa torchmetrics 0.7.2 y pytorch-lightning 1.4.8, más viejos de lo que asume el código.
p1 = f"{REPO}/hyperncd/modules/Discoverer_hyper.py"
s = open(p1, encoding="utf-8").read()
if "jaccard_index as _jaccard_index" in s:
    print("Discoverer_hyper.py: ya estaba parchado")
else:
    a = "from torchmetrics.functional.classification import multiclass_jaccard_index\n"
    b = ("try:\n    from torchmetrics.functional.classification import multiclass_jaccard_index\nexcept ImportError:\n"
         "    from torchmetrics.functional import jaccard_index as _jaccard_index\n\n"
         "    def multiclass_jaccard_index(preds, target, num_classes, average=\"none\"):\n"
         "        return _jaccard_index(preds, target, num_classes=num_classes, reduction=\"none\")\n")
    assert a in s, "no encuentro la importación en Discoverer_hyper.py"
    open(p1, "w", encoding="utf-8").write(s.replace(a, b, 1)); print("Discoverer_hyper.py: ✅ parchado")

p2 = f"{REPO}/hyperncd/train.py"
s = open(p2, encoding="utf-8").read()
if "fit_nuevo" in s:
    print("train.py: ya estaba parchado")
else:
    ini = "    trainer = pl.Trainer(\n        max_epochs=args.epochs,"
    fin = "trainer.fit(model, ckpt_path=args.resume)\n"
    assert ini in s and fin in s, "no encuentro el bloque del Trainer en train.py"
    i = s.index(ini); j = s.index(fin, i) + len(fin)
    nuevo = ("    trainer_kw = dict(max_epochs=args.epochs, logger=loggers, accelerator=\"gpu\", devices=1,\n"
             "                      num_sanity_val_steps=0, callbacks=callbacks)\n"
             "    import inspect\n    fit_nuevo = \"ckpt_path\" in inspect.signature(pl.Trainer.fit).parameters\n"
             "    if args.resume and not fit_nuevo:\n        trainer_kw[\"resume_from_checkpoint\"] = args.resume\n"
             "    trainer = pl.Trainer(**trainer_kw)\n"
             "    if fit_nuevo:\n        trainer.fit(model, ckpt_path=args.resume)\n    else:\n        trainer.fit(model)\n")
    open(p2, "w", encoding="utf-8").write(s[:i] + nuevo + s[j:]); print("train.py: ✅ parchado")
assert "ArCH" in open(f"{REPO}/hyperncd/utils/dataset.py").read(), "dataset.py sin ArCH: sube los cambios a GitHub"


In [ ]:
# @title 3. Restaurar hyperncd-env + MinkowskiEngine (de tu Drive) e instalar wandb
%%bash
set -e

ENV_CACHE="/content/drive/MyDrive/PFC1_HyperNCD_Sonata/entornos/hyperncd_env_cache.tar.gz"

if [ ! -d "/content/miniconda3" ]; then
    echo "[1/4] Instalando Miniconda3 base..."
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    bash /tmp/miniconda.sh -b -p /content/miniconda3
fi

source /content/miniconda3/etc/profile.d/conda.sh
conda config --set always_yes yes
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true

# Si ya existe la imagen del entorno compilada en tu Google Drive, la restauramos en ~20 segundos
if [ -f "$ENV_CACHE" ] && [ ! -d "/content/miniconda3/envs/hyperncd-env" ]; then
    echo "⚡ Encontrado entorno pre-compilado en tu Drive ($ENV_CACHE). Restaurando en ~20 segundos..."
    mkdir -p /content/miniconda3/envs/hyperncd-env
    tar -xzf "$ENV_CACHE" -C /content/miniconda3/envs/hyperncd-env
    /content/miniconda3/envs/hyperncd-env/bin/python -c "import torch, MinkowskiEngine as ME; print('✅ Entorno restaurado desde Drive en segundos! MinkowskiEngine:', ME.__version__)"
    exit 0
fi

if [ ! -d "/content/miniconda3/envs/hyperncd-env" ]; then
    echo "[2/4] Creando entorno hyperncd-env (Python 3.8 + CUDA 11.8 + OpenBLAS)..."
    conda create -y -n hyperncd-env python=3.8 -c conda-forge
    conda install -y -n hyperncd-env -c nvidia/label/cuda-11.8.0 cuda-nvcc cuda-cudart-dev cuda-libraries-dev
    conda install -y -n hyperncd-env -c conda-forge openblas-devel ninja gxx_linux-64=11
fi

conda activate hyperncd-env

echo "[3/4] Instalando PyTorch 1.10.1+cu113 y dependencias de HyperNCD..."
pip install -q torch==1.10.1+cu113 torchvision==0.11.2+cu113 -f https://download.pytorch.org/whl/torch_stable.html
pip install -q pytorch-lightning==1.4.8 torchmetrics==0.7.2 scipy==1.7.3 pyyaml tqdm scikit-learn plotly matplotlib
pip install -q torch-scatter -f https://data.pyg.org/whl/torch-1.10.0+cu113.html

if ! python -c "import MinkowskiEngine" 2>/dev/null; then
    echo "[4/4] Compilando e instalando MinkowskiEngine (solo esta primera vez)..."
    # Asegurar cabecera nvtx3/nvToolsExt.h que pide MinkowskiEngine
    conda install -y -n hyperncd-env -c nvidia/label/cuda-11.8.0 cuda-nvtx-dev 2>/dev/null || true
    if [ ! -f "$CONDA_PREFIX/include/nvtx3/nvToolsExt.h" ]; then
        git clone -q --depth 1 https://github.com/NVIDIA/NVTX.git /tmp/NVTX
        cp -r /tmp/NVTX/c/include/nvtx3 "$CONDA_PREFIX/include/"
        rm -rf /tmp/NVTX
    fi
    if [ ! -d "/content/MinkowskiEngine" ]; then
        git clone -q https://github.com/NVIDIA/MinkowskiEngine.git /content/MinkowskiEngine
    fi
    cd /content/MinkowskiEngine
    export CUDA_HOME="$CONDA_PREFIX"
    export PATH="$CONDA_PREFIX/bin:$PATH"
    export CPLUS_INCLUDE_PATH="$CONDA_PREFIX/include:${CPLUS_INCLUDE_PATH:-}"
    export TORCH_CUDA_ARCH_LIST="7.5;8.0;8.6"
    export MAX_JOBS=4
    export CC="$CONDA_PREFIX/bin/x86_64-conda-linux-gnu-gcc"
    export CXX="$CONDA_PREFIX/bin/x86_64-conda-linux-gnu-g++"
    python setup.py install --blas_include_dirs=${CONDA_PREFIX}/include --blas=openblas
fi

python -c "import torch, MinkowskiEngine as ME; print('✅ MinkowskiEngine listo, versión:', ME.__version__, '| GPU:', torch.cuda.get_device_name(0))"

# Guardar imagen del entorno ya compilado en tu Google Drive para no volver a compilar nunca más
if [ -d "/content/drive/MyDrive" ] && [ ! -f "$ENV_CACHE" ]; then
    echo "💾 Guardando imagen del entorno compilado en tu Drive ($ENV_CACHE) para que la próxima vez cargue en 20 segundos..."
    mkdir -p "$(dirname "$ENV_CACHE")"
    tar -czf "$ENV_CACHE" -C /content/miniconda3/envs/hyperncd-env .
    echo "✅ Imagen del entorno guardada en Drive."
fi

In [ ]:
# @title 3b. wandb en el entorno (train.py lo importa; se usa en modo offline)
!/content/miniconda3/envs/hyperncd-env/bin/pip install -q wandb
!/content/miniconda3/envs/hyperncd-env/bin/python -c "import distutils.version, wandb, MinkowskiEngine as ME, pytorch_lightning as pl, torch; print('wandb', wandb.__version__, '| ME', ME.__version__, '| PL', pl.__version__, '| torch', torch.__version__, '| GPU', torch.cuda.is_available())"

In [ ]:
# @title 4. Convertir ArCH a bloques (una sola vez; reanudable; sin GPU)
MON = f"{REPO}/experimentacion/monitor_recursos/monitor_recursos.py"
!pip -q install psutil
!python {MON} --salida {SAL}/medicion/conversion --nombre conversion_ArCH --cada 10 -- \
    python {REPO}/experimentacion/arch_entrenamiento/convertir_arch.py --entrada {CRUDO} --salida {TILES}

In [ ]:
# @title 5. PRUEBA de 1 época, medida (decide con esto cuánto durará la corrida completa)
import subprocess
PY = "/content/miniconda3/envs/hyperncd-env/bin/python"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["ARCH_TILES"] = TILES
os.environ["HYPERNCD_CACHE"] = f"{ARCH}/cache_regiones"     # regiones (DBSCAN) cacheadas en Drive
HIPER = [PY, "train.py", "-s", "0", "--dataset", "ArCH", "--offline", "--use_scheduler",
         "--batch_size", str(BATCH), "--num_workers", str(WORKERS),
         "--lam", "1", "--lam_region", "1", "--gamma", "1", "--alpha", "1", "--gamma_decrease", "0.5",
         "--smooth_bound", "10", "--ak_bound", "0.005", "--dbscan", "0.5",
         "--log_dir", f"{SAL}/logs", "--ckpt_every_steps", "100"]
def lanzar(nombre, epocas, extra=()):
    cmd = ["python", MON, "--salida", f"{SAL}/medicion/{nombre}", "--nombre", nombre, "--cada", "5", "--",
           *HIPER, "--epochs", str(epocas), "--comment", nombre, "--checkpoint_dir", f"{SAL}/checkpoints", *extra]
    p = subprocess.Popen(cmd, cwd=f"{REPO}/hyperncd", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linea in p.stdout:          # sin esto, Colab no muestra la salida (ni los errores) del comando
        print(linea, end="")
    return p.wait()
print("código de salida:", lanzar("prueba_1epoca", 1))

In [ ]:
# @title 5b. Ver la medición de la prueba (tiempo, RAM, GPU)
import json
from IPython.display import Image, display
r = json.load(open(f"{SAL}/medicion/prueba_1epoca/resumen.json"))
print(json.dumps(r, indent=1, ensure_ascii=False))
display(Image(f"{SAL}/medicion/prueba_1epoca/recursos.png"))
print(f"\n≈ {r['duracion_min']} min por época  ->  {EPOCAS} épocas ≈ {r['duracion_min'] * EPOCAS / 60:.1f} h")

In [ ]:
# @title 6. Entrenamiento completo, medido (reanuda solo si ya hay last.ckpt)
import glob
ult = sorted(glob.glob(f"{SAL}/checkpoints/ArCH/corrida_completa/last.ckpt"))
extra = ["--resume", ult[-1]] if ult else []
print("reanudando desde:" if ult else "empezando de cero", ult[-1] if ult else "")
print("código de salida:", lanzar("corrida_completa", EPOCAS, extra))

In [ ]:
# @title 7. Resultados del entrenamiento: medición y métricas por época
import json, glob, os
import pandas as pd
from IPython.display import Image, display
M = f"{SAL}/medicion/corrida_completa"
if os.path.exists(f"{M}/resumen.json"):
    print(json.dumps(json.load(open(f"{M}/resumen.json")), indent=1, ensure_ascii=False)); display(Image(f"{M}/recursos.png"))
csvs = glob.glob(f"{SAL}/logs/**/metrics.csv", recursive=True)
if csvs:
    d = pd.read_csv(sorted(csvs, key=os.path.getmtime)[-1])
    col = [c for c in d.columns if c.startswith("valid/") or c in ("epoch", "train/mIoU")]
    print(d[col].dropna(how="all", subset=[c for c in col if c != "epoch"]).tail(15).to_string())


In [ ]:
# @title 8. Predecir la capilla etiquetada con el MEJOR checkpoint y con el ÚLTIMO (mide el dominio)
import re, glob, os, subprocess
PY = "/content/miniconda3/envs/hyperncd-env/bin/python"
CAP = f"{DRIVE}/m1/salida/capilla_etiquetada/capilla_etiquetada.npy"
assert os.path.exists(CAP), f"Falta {CAP}: corre el cuaderno 1"
CK = f"{SAL}/checkpoints/ArCH/corrida_completa"
mejores = glob.glob(f"{CK}/ArCH_S0_best-epoch=*.ckpt")
ep = lambda c: int(re.search(r"epoch=(\d+)", c).group(1))
CKPTS = {}
if mejores: CKPTS["best"] = max(mejores, key=ep)
if os.path.exists(f"{CK}/last.ckpt"): CKPTS["last"] = f"{CK}/last.ckpt"
print("checkpoints:", CKPTS)
assert CKPTS, f"No hay checkpoints en {CK}: ¿terminó la celda 6?"
PRED = f"{SAL}/predicciones"; os.makedirs(PRED, exist_ok=True)
for nombre, ck in CKPTS.items():
    print("\n" + "=" * 60 + f"\n  capilla_etiquetada con {nombre}: {os.path.basename(ck)}\n" + "=" * 60, flush=True)
    cmd = ["python", MON, "--salida", f"{SAL}/medicion/pred_capilla_{nombre}", "--nombre", f"pred_capilla_{nombre}", "--cada", "3", "--",
           PY, "predecir_hyperncd.py", "--input", CAP, "--ckpt", ck, "--output", f"{PRED}/capilla_etiquetada_{nombre}_fsem.npz"]
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linea in p.stdout: print(linea, end="")
    print("código de salida:", p.wait())


In [ ]:
# @title 9. Empaquetar logs, predicciones y mediciones para bajarlos (y calcular las métricas en local)
import zipfile, glob, os
from google.colab import files
INCLUIR_CKPT = False        # @param {type:"boolean"}  (el mejor .ckpt pesa ~450 MB; ya queda en tu Drive)
zp = "/content/entrenamiento_arch_resultados.zip"
validos = (".png", ".json", ".csv", ".md", ".yaml", ".npz", ".txt")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for base in (f"{SAL}/medicion", f"{SAL}/logs", f"{SAL}/predicciones"):
        for raiz, _, fs in os.walk(base):
            for n in fs:
                if n.endswith(validos) and "wandb" not in raiz:
                    z.write(os.path.join(raiz, n), os.path.relpath(os.path.join(raiz, n), SAL))
    if os.path.exists(f"{TILES}/resumen_bloques.json"): z.write(f"{TILES}/resumen_bloques.json", "resumen_bloques.json")
    if INCLUIR_CKPT:
        for ck in glob.glob(f"{SAL}/checkpoints/ArCH/corrida_completa/*.ckpt"): z.write(ck, "checkpoints/" + os.path.basename(ck))
print(round(os.path.getsize(zp) / 2**20, 1), "MB")
files.download(zp)
